# Capítulo 3 — Escenarios de Usuario y Page Object Model

> *«El Page Object Model no es un patrón para ocultar Playwright; es un patrón para que los tests hablen el lenguaje del usuario, no el del DOM.»*

**Prerequisito:** Haber completado los Capítulos 1 y 2.

**Objetivo:** Aprender a modelar flujos de usuario reales como tests E2E, y organizar el código de pruebas usando el patrón **Page Object Model (POM)** para maximizar la mantenibilidad y reutilización.

---

## 1. ¿Qué es un escenario de usuario?

Un **escenario de usuario** describe una secuencia de acciones que un usuario real realizaría para completar un objetivo. En las pruebas E2E, cada escenario debería corresponderse con un caso de uso de negocio.

### Ejemplo de escenario bien descrito

**Escenario:** «Como usuario, quiero crear una tarea, completarla y verificar que queda registrada como completada para poder hacer seguimiento de mi progreso.»

En formato Gherkin (BDD):
```gherkin
Given que estoy en la página principal del gestor de tareas
  And la lista de tareas está vacía
When escribo "Comprar pan" en el campo de nueva tarea
  And hago clic en el botón "Agregar"
Then la tarea "Comprar pan" debe aparecer en la lista
  And el botón "Completar" debe estar visible junto a ella
When hago clic en "Completar" de la tarea "Comprar pan"
Then la tarea debe mostrar el badge "✓ Completada"
  And el botón "Completar" ya no debe estar visible
```

Este nivel de detalle es lo que debería reflejar nuestro test E2E.

---

## 2. El problema sin Page Object Model

Imagina que tienes 15 tests E2E y todos repiten:
```python
page.fill("[data-testid='input-titulo']", titulo)
page.click("[data-testid='btn-agregar']")
page.wait_for_load_state("networkidle")
```

Ahora el equipo decide renombrar el `data-testid` de `input-titulo` a `task-title-input`. Tienes que cambiar **15 tests**. Esto viola el principio **DRY** (Don't Repeat Yourself) y hace la suite muy frágil.

**Consecuencias del código sin POM:**
- Un cambio de selector rompe múltiples archivos.
- Los tests mezclan los *detalles de UI* con la *lógica del escenario*.
- Es difícil leer el test y entender qué flujo de usuario está verificando.

---

## 3. Page Object Model (POM)

El **Page Object Model** encapsula la interacción con una página web en una clase. Cada método representa una acción o consulta sin exponer detalles de implementación.

```
Test                         Page Object
─────────────────────────    ────────────────────────────────
task_page.crear_tarea("X") →  fill(input) + click(button)
task_page.completar()      →  click(btn-completar) + wait
task_page.contar_tareas()  →  locator.count()
```

**Beneficios:**
1. **Un solo lugar de cambio:** Si cambia un selector, solo se modifica el Page Object.
2. **Tests más legibles:** Expresan el escenario, no los detalles de la UI.
3. **Reutilización:** El mismo POM sirve para múltiples tests.
4. **Separación de responsabilidades:** El test dice *qué* hacer; el POM dice *cómo*.

---

## 4. Implementación: TaskPage

In [ ]:
# ──────────────────────────────────────────────────────────────
# PAGE OBJECT MODEL: TaskPage
# Guarda este código en tests/page_objects.py
# ──────────────────────────────────────────────────────────────
from playwright.sync_api import Page, expect


class TaskPage:
    """
    Page Object para el Gestor de Tareas.
    Los tests no deben conocer los data-testid directamente.
    """

    def __init__(self, page: Page, base_url: str = "http://localhost:5000"):
        self._page = page
        self._base_url = base_url

    def navegar(self):
        self._page.goto(self._base_url)

    @property
    def input_titulo(self):
        return self._page.locator("[data-testid='input-titulo']")

    @property
    def btn_agregar(self):
        return self._page.locator("[data-testid='btn-agregar']")

    @property
    def lista_tareas(self):
        return self._page.locator("[data-testid='tarea-item']")

    @property
    def mensaje_lista_vacia(self):
        return self._page.locator("[data-testid='msg-lista-vacia']")

    def crear_tarea(self, titulo: str):
        """Crea una nueva tarea con el título dado."""
        self.input_titulo.fill(titulo)
        self.btn_agregar.click()
        self._page.wait_for_load_state("networkidle")

    def completar_tarea(self, indice: int = 0):
        """Completa la tarea en la posición indicada."""
        self._page.locator("[data-testid='btn-completar']").nth(indice).click()
        self._page.wait_for_load_state("networkidle")

    def eliminar_tarea(self, indice: int = 0):
        """Elimina la tarea en la posición indicada."""
        self._page.locator("[data-testid='btn-eliminar']").nth(indice).click()
        self._page.wait_for_load_state("networkidle")

    def contar_tareas(self) -> int:
        return self.lista_tareas.count()

    def titulo_de_tarea(self, indice: int = 0) -> str:
        return self._page.locator("[data-testid='tarea-titulo']").nth(indice).inner_text()

    def esta_completada(self, indice: int = 0) -> bool:
        return self._page.locator("[data-testid='badge-completada']").nth(indice).is_visible()

    def lista_vacia(self) -> bool:
        return self.mensaje_lista_vacia.is_visible()

    def titulos_de_todas_las_tareas(self) -> list:
        items = self._page.locator("[data-testid='tarea-titulo']")
        return [items.nth(i).inner_text() for i in range(items.count())]


print("Clase TaskPage definida.")

---

## 5. Los 5 escenarios ejecutables del taller

Cada escenario está escrito usando el POM. Observa que los tests describen el *comportamiento del usuario*, no los selectores.

In [ ]:
# ── Setup común ────────────────────────────────────────────────
from playwright.sync_api import sync_playwright, expect
import urllib.request

BASE_URL = "http://localhost:5000"

def limpiar_estado():
    urllib.request.urlopen(
        urllib.request.Request(f"{BASE_URL}/tasks/clear", method="POST")
    )

print("Setup listo.")

In [ ]:
# ESCENARIO 1: Crear una tarea y verificar que aparece en la lista
with sync_playwright() as pw:
    browser = pw.chromium.launch(headless=True)
    page = browser.new_page()
    tp = TaskPage(page, BASE_URL)
    limpiar_estado()
    tp.navegar()

    # Given: la lista está vacía
    assert tp.lista_vacia(), "Precondición: la lista debe estar vacía"

    # When: el usuario crea una tarea
    tp.crear_tarea("Comprar pan")

    # Then: la tarea aparece en la lista con el título correcto
    assert tp.contar_tareas() == 1
    assert tp.titulo_de_tarea(0) == "Comprar pan"
    assert not tp.lista_vacia()

    print("✓ Escenario 1: Crear tarea — PASÓ")
    browser.close()

In [ ]:
# ESCENARIO 2: Completar una tarea y verificar el badge
with sync_playwright() as pw:
    browser = pw.chromium.launch(headless=True)
    page = browser.new_page()
    tp = TaskPage(page, BASE_URL)
    limpiar_estado()
    tp.navegar()
    tp.crear_tarea("Leer un libro")

    # Given: existe una tarea pendiente
    assert not tp.esta_completada(0)

    # When: el usuario completa la tarea
    tp.completar_tarea(0)

    # Then: la tarea muestra el badge y el botón completar desaparece
    assert tp.esta_completada(0), "Debe mostrarse el badge 'Completada'"
    expect(page.locator("[data-testid='btn-completar']")).to_have_count(0)

    print("✓ Escenario 2: Completar tarea — PASÓ")
    browser.close()

In [ ]:
# ESCENARIO 3: Eliminar una tarea
with sync_playwright() as pw:
    browser = pw.chromium.launch(headless=True)
    page = browser.new_page()
    tp = TaskPage(page, BASE_URL)
    limpiar_estado()
    tp.navegar()
    tp.crear_tarea("Tarea a eliminar")
    assert tp.contar_tareas() == 1

    # When: el usuario elimina la tarea
    tp.eliminar_tarea(0)

    # Then: la lista queda vacía
    assert tp.contar_tareas() == 0
    assert tp.lista_vacia()

    print("✓ Escenario 3: Eliminar tarea — PASÓ")
    browser.close()

In [ ]:
# ESCENARIO 4: Múltiples tareas — orden y gestión independiente
with sync_playwright() as pw:
    browser = pw.chromium.launch(headless=True)
    page = browser.new_page()
    tp = TaskPage(page, BASE_URL)
    limpiar_estado()
    tp.navegar()

    titulos = ["Tarea Alpha", "Tarea Beta", "Tarea Gamma"]
    for titulo in titulos:
        tp.crear_tarea(titulo)

    assert tp.contar_tareas() == 3
    assert tp.titulos_de_todas_las_tareas() == titulos

    # When: se completa solo la segunda tarea
    tp.completar_tarea(1)

    # Then: solo la segunda está marcada
    assert not tp.esta_completada(0)
    assert tp.esta_completada(1)
    assert not tp.esta_completada(2)

    print("✓ Escenario 4: Múltiples tareas — PASÓ")
    browser.close()

In [ ]:
# ESCENARIO 5: Flujo completo Crear → Completar → Eliminar
with sync_playwright() as pw:
    browser = pw.chromium.launch(headless=True)
    page = browser.new_page()
    tp = TaskPage(page, BASE_URL)
    limpiar_estado()
    tp.navegar()

    tp.crear_tarea("Flujo completo")
    assert tp.contar_tareas() == 1
    print("  Paso 1 ✓ Tarea creada")

    tp.completar_tarea(0)
    assert tp.esta_completada(0)
    print("  Paso 2 ✓ Tarea completada")

    tp.eliminar_tarea(0)
    assert tp.contar_tareas() == 0
    assert tp.lista_vacia()
    print("  Paso 3 ✓ Tarea eliminada")

    print("✓ Escenario 5: Flujo completo — PASÓ")
    browser.close()

---

## 6. Mejoras avanzadas al POM

### 6.1 POM con fixtures de pytest

```python
# En conftest.py
@pytest.fixture
def task_page(page, live_server):
    return TaskPage(page, live_server)

# En test_tareas_e2e.py
def test_crear_tarea(task_page):
    task_page.crear_tarea("Mi tarea")
    assert task_page.contar_tareas() == 1
```

### 6.2 Component Objects para elementos repetitivos

Cuando un elemento se repite (ej: una tarjeta de tarea), se puede crear un Component Object:

```python
class TaskCard:
    def __init__(self, locator):
        self._loc = locator  # li[data-testid='tarea-item']

    def titulo(self) -> str:
        return self._loc.locator("[data-testid='tarea-titulo']").inner_text()

    def esta_completada(self) -> bool:
        return self._loc.locator("[data-testid='badge-completada']").is_visible()

    def completar(self):
        self._loc.locator("[data-testid='btn-completar']").click()

    def eliminar(self):
        self._loc.locator("[data-testid='btn-eliminar']").click()
```

### 6.3 Page Objects jerárquicos

Para aplicaciones grandes:
```python
class TaskApp:
    def __init__(self, page):
        self.navbar = NavBarComponent(page)
        self.task_list = TaskListComponent(page)
        self.form = NewTaskFormComponent(page)
```

---

## 7. Preguntas para el informe

1. **POM vs código directo:** Reescribe el Escenario 1 sin usar `TaskPage`. Compara los dos códigos — ¿cuál es más legible? ¿cuál es más fácil de mantener?

2. **Traducción Gherkin → test:** Implementa el siguiente escenario como test E2E:
   ```gherkin
   Given que existen 2 tareas
   When elimino la primera
   Then solo debe quedar 1 tarea
   And el título de la restante debe ser el de la segunda
   ```

3. **Aislamiento:** ¿Por qué es fundamental llamar a `limpiar_estado()` antes de cada escenario? ¿Qué fallaría si se omitiera?

4. **Component Objects:** Implementa la clase `TaskCard` y reescribe el Escenario 4 usándola.